# 🏢 Khám Phá, Đánh Giá & Làm Sạch Bảng: `companies`

Notebook này là quy trình độc lập chuyên biệt cho **Bảng 1 — `companies` (Dimension Doanh Nghiệp)** theo chuẩn schema PostgreSQL:

1. 📂 **Nạp & Kết Hợp Dữ Liệu Nguồn**: Trích xuất các trường doanh nghiệp từ Bronze Listings và Details.
2. 🔍 **Khám Phá & Đánh Giá Chất Lượng (EDA & Profiling)**: Khảo sát độ khuyết thiếu, các biến thể tên công ty, định dạng quy mô, ngành nghề.
3. 🧹 **Xây Dựng Logic Làm Sạch & Chuẩn Hóa**:
   - Loại bỏ tiền tố pháp lý (*Công ty Cổ phần, TNHH...*) đưa về tên nhận diện thương hiệu `company_name_normalized`.
   - Nhận diện `website_domain` (phân biệt website riêng của công ty với trang tuyển dụng trên sàn `topcv.vn`, `vietnamworks.com`).
   - Bóc tách quy mô thành số nguyên `company_size_min` và `company_size_max`.
   - Chuẩn hóa thành phố trụ sở `company_city` và ngành nghề `industry_normalized`.
4. 🔄 **Khử Trùng Lặp Thực Thể (Entity Deduplication)**: Gom nhóm các tin tuyển dụng cùng công ty về 1 bản ghi duy nhất, cấp phát `company_id`.
5. 📊 **Đánh Giá Kết Quả & Preview Bảng `companies` Hoàn Chỉnh**.

In [2]:
import os
import re
import json
from urllib.parse import urlparse
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import display

# Cấu hình pandas hiển thị đầy đủ
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('🚀 Môi trường xử lý bảng companies đã sẵn sàng!')

🚀 Môi trường xử lý bảng companies đã sẵn sàng!


## 1. Nạp Dữ Liệu Doanh Nghiệp Từ Tầng Master Bronze

Chúng ta kết hợp thông tin doanh nghiệp từ cả 2 bảng Bronze:
- `bronze_details_valid.json`: Chứa `company_name_raw`, `company_url`, `company_size_raw`, `company_location_raw`, `industry_raw`.
- `bronze_listings_master.json`: Bổ sung `company_logo_url`.

In [4]:
details_path = os.path.abspath('../../data/bronze/bronze_details_valid.json')
if not os.path.exists(details_path):
    details_path = os.path.abspath('data/bronze/bronze_details_valid.json')

listings_path = os.path.abspath('../../data/bronze/bronze_listings_master.json')
if not os.path.exists(listings_path):
    listings_path = os.path.abspath('data/bronze/bronze_listings_master.json')

with open(details_path, 'r', encoding='utf-8') as f:
    details = json.load(f)
with open(listings_path, 'r', encoding='utf-8') as f:
    listings = json.load(f)

df_det = pd.DataFrame(details)
df_list = pd.DataFrame(listings)

# Trích xuất các trường công ty và ghép logo từ listing
company_raw_cols = ['source', 'source_job_id', 'company_name_raw', 'company_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
df_comp_base = df_det[company_raw_cols].copy()

# Ghép company_logo_url từ listing
df_logos = df_list[['source', 'source_job_id', 'company_logo_url']].drop_duplicates(subset=['source', 'source_job_id'])
df_comp_merged = pd.merge(df_comp_base, df_logos, on=['source', 'source_job_id'], how='left')

print(f'📊 Tổng số bản ghi công việc chứa thông tin công ty: {len(df_comp_merged):,} dòng')
display(df_comp_merged[['company_name_raw', 'company_url', 'company_size_raw', 'industry_raw', 'company_logo_url']].head(5))

📊 Tổng số bản ghi công việc chứa thông tin công ty: 2,327 dòng
                               company_name_raw  \
0                   Công ty TNHH ZIGExN VeNtura   
1                      Công ty TNHH Điện tử R&P   
2                         CÔNG TY TNHH FOXSCORE   
3  CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL   
4                CÔNG TY TNHH VODAPLAY VIỆT NAM   

                                                                       company_url  \
0               https://www.topcv.vn/cong-ty/cong-ty-tnhh-zigexn-ventura/1036.html   
1                  https://www.topcv.vn/cong-ty/cong-ty-tnhh-dien-tu-rp/19536.html   
2                   https://www.topcv.vn/cong-ty/cong-ty-tnhh-foxscore/249702.html   
3  https://www.topcv.vn/cong-ty/cong-ty-co-phan-thuong-mai-quoc-te-lb-global/18...   
4          https://www.topcv.vn/cong-ty/cong-ty-tnhh-vodaplay-viet-nam/107235.html   

    company_size_raw                                          industry_raw  \
0  100-499 nhân viên               

## 2. Khám Phá & Đánh Giá Chất Lượng Dữ Liệu Công Ty (Data Profiling & EDA)

Khảo sát các vấn đề thực tế trong dữ liệu thô:
1. **Tỷ lệ khuyết thiếu (Missing / Null / Empty)** trên từng trường của công ty.
2. **Các biến thể tên công ty**: Sự xuất hiện của các tiền tố pháp lý rườm rà (*CÔNG TY TNHH, CỔ PHẦN, CHI NHÁNH, TẬP ĐOÀN...*).
3. **Định dạng quy mô nhân sự thô**: Khoảng số, dấu cộng (+), hay chuỗi văn bản.

In [6]:
# 1. Đánh giá tỷ lệ khuyết thiếu của các trường công ty
comp_fields = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw']
null_cnt = df_comp_merged[comp_fields].isnull().sum()
empty_cnt = (df_comp_merged[comp_fields] == '').sum()
tot_miss = null_cnt + empty_cnt
miss_pct = (tot_miss / len(df_comp_merged) * 100).round(2)

df_comp_missing = pd.DataFrame({
    'Số Lượng Rỗng/Khuyết': tot_miss,
    'Tỷ Lệ Khuyết (%)': miss_pct,
    'Tỷ Lệ Đầy Đủ (%)': (100 - miss_pct).round(2)
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('📊 1. BẢNG KIỂM KÊ TỶ LỆ KHUYẾT THIẾU CÁC TRƯỜNG CÔNG TY:')
display(df_comp_missing)

# 2. Khảo sát các dạng quy mô công ty phổ biến nhất
print('\n📊 2. TOP 10 ĐỊNH DẠNG QUY MÔ NHÂN SỰ GỐC (company_size_raw):')
display(df_comp_merged['company_size_raw'].replace('', '(Để trống)').value_counts().head(10))

# 3. Khảo sát các biến thể tiền tố tên công ty
sample_names = df_comp_merged['company_name_raw'].dropna().unique()[:10]
print('\n🏢 3. MẪU TÊN CÔNG TY GỐC TRÊN SÀN (company_name_raw):')
for name in sample_names:
    print(f'  • "{name}"')

📊 1. BẢNG KIỂM KÊ TỶ LỆ KHUYẾT THIẾU CÁC TRƯỜNG CÔNG TY:
                      Số Lượng Rỗng/Khuyết  Tỷ Lệ Khuyết (%)  Tỷ Lệ Đầy Đủ (%)
company_size_raw                       623             26.77             73.23
company_logo_url                        25              1.07             98.93
company_url                              0              0.00            100.00
company_name_raw                         0              0.00            100.00
company_location_raw                     0              0.00            100.00
industry_raw                             0              0.00            100.00

📊 2. TOP 10 ĐỊNH DẠNG QUY MÔ NHÂN SỰ GỐC (company_size_raw):
company_size_raw
(Để trống)            623
100-499 nhân viên     385
25-99 nhân viên       362
100-499               167
1.000-4.999           111
500-1000 nhân viên    103
1000+ nhân viên        95
10-24 nhân viên        90
500-999                62
25-99                  57
Name: count, dtype: int64

🏢 3. MẪU TÊN CÔNG TY GỐC

## 3. Xây Dựng Logic Làm Sạch & Chuẩn Hóa Cho Bảng `companies`

Thiết kế các hàm xử lý chuyên biệt:
1. **`clean_company_name()`**: Loại bỏ các tiền tố/hậu tố pháp lý (*Công ty TNHH MTV, CÔNG TY CỔ PHẦN, Chi nhánh, Corp, JSC, Co., Ltd*) để đưa về tên nhận diện thương hiệu chuẩn.
2. **`extract_website_domain()`**: Phân biệt website riêng của doanh nghiệp với URL trang hồ sơ trên sàn (`topcv.vn`, `vietnamworks.com`, `topdev.vn`).
3. **`parse_company_size()`**: Tách chuỗi quy mô thành `company_size_min` và `company_size_max` (kiểu số nguyên `INTEGER`).
4. **`normalize_company_city()`**: Chuẩn hóa thành phố của trụ sở công ty.

In [8]:
# --- BỘ HÀM CHUẨN HÓA DOANH NGHIỆP ---

PORTAL_DOMAINS = {'topcv.vn', 'vietnamworks.com', 'topdev.vn', 'saramin.vn'}

def clean_company_name(name):
    if not name or pd.isna(name):
        return "Unknown Company"
    name = str(name).strip()
    prefixes = [
        r"^CÔNG TY TNHH MTV\s+", r"^CÔNG TY TNHH\s+", r"^Công ty TNHH MTV\s+", r"^Công ty TNHH\s+",
        r"^CÔNG TY CỔ PHẦN\s+", r"^Công ty Cổ phần\s+", r"^Công Ty Cổ Phần\s+", r"^Công Ty TNHH\s+",
        r"^TẬP ĐOÀN\s+", r"^Tập đoàn\s+", r"^TỔNG CÔNG TY\s+", r"^Tổng Công ty\s+",
        r"^Chi nhánh\s+", r"^CHI NHÁNH\s+",
    ]
    for p in prefixes:
        name = re.sub(p, "", name, flags=re.IGNORECASE)
    suffixes = [
        r"\s+-\s+Chi nhánh.*$", r"\s+-\s+CN.*$", r"\s+JSC$", r"\s+Co\.,\s*Ltd\.?$",
    ]
    for s in suffixes:
        name = re.sub(s, "", name, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', name).strip(' -|/,')

def extract_website_domain(url_str):
    if not url_str or pd.isna(url_str):
        return None
    u = str(url_str).strip()
    if not u.startswith(('http://', 'https://')):
        u = 'https://' + u
    try:
        parsed = urlparse(u)
        domain = parsed.netloc.lower()
        if domain.startswith('www.'):
            domain = domain[4:]
        # Nếu URL trỏ về hồ sơ sàn thì không phải website riêng của công ty
        if domain in PORTAL_DOMAINS:
            return None
        return domain if '.' in domain else None
    except Exception:
        return None

def parse_company_size(size_str):
    if not size_str or pd.isna(size_str):
        return None, None
    s = str(size_str).strip()
    s_clean = s.replace('.', '')
    range_match = re.search(r"(\d+)\s*[-–]\s*(\d+)", s_clean)
    if range_match:
        return int(range_match.group(1)), int(range_match.group(2))
    plus_match = re.search(r"(\d+)\s*\+", s_clean)
    if plus_match:
        return int(plus_match.group(1)), None
    more_match = re.search(r"(?:hơn|trên)\s*(\d+)", s_clean, re.IGNORECASE)
    if more_match:
        return int(more_match.group(1)), None
    less_match = re.search(r"(?:ít hơn|dưới|<\s*)(\d+)", s_clean, re.IGNORECASE)
    if less_match:
        return 1, int(less_match.group(1))
    return None, None

def normalize_company_city(loc_str, comp_loc_str=""):
    combined = (str(comp_loc_str) + ' ' + str(loc_str)).lower()
    if any(k in combined for k in ['hồ chí minh', 'tp.hcm', 'hcm', 'sài gòn', 'saigon']):
        return 'Hồ Chí Minh'
    if any(k in combined for k in ['hà nội', 'ha noi', 'hn']):
        return 'Hà Nội'
    if any(k in combined for k in ['đà nẵng', 'da nang']):
        return 'Đà Nẵng'
    if any(k in combined for k in ['hải phòng', 'hai phong']):
        return 'Hải Phòng'
    if any(k in combined for k in ['cần thơ', 'can tho']):
        return 'Cần Thơ'
    return 'Khác / Chưa rõ'

# Thử nghiệm nhanh các hàm làm sạch
print('🧪 KIỂM TRA THỬ NGHIỆM CÁC HÀM LÀM SẠCH:')
print('  • Tên gốc: "CÔNG TY TNHH ZIGExN VeNtura - Chi nhánh HCM"')
print('    -> Tên sạch:', clean_company_name('CÔNG TY TNHH ZIGExN VeNtura - Chi nhánh HCM'))
print('  • Quy mô gốc: "100-499 nhân viên" -> min/max:', parse_company_size('100-499 nhân viên'))
print('  • Quy mô gốc: "1.000-4.999"        -> min/max:', parse_company_size('1.000-4.999'))
print('  • Quy mô gốc: "1000+ nhân viên"   -> min/max:', parse_company_size('1000+ nhân viên'))
print('  • URL riêng: "https://careers.fpt.com/about" -> domain:', extract_website_domain('https://careers.fpt.com/about'))
print('  • URL sàn  : "https://www.topcv.vn/cong-ty/elva" -> domain:', extract_website_domain('https://www.topcv.vn/cong-ty/elva'))

🧪 KIỂM TRA THỬ NGHIỆM CÁC HÀM LÀM SẠCH:
  • Tên gốc: "CÔNG TY TNHH ZIGExN VeNtura - Chi nhánh HCM"
    -> Tên sạch: ZIGExN VeNtura
  • Quy mô gốc: "100-499 nhân viên" -> min/max: (100, 499)
  • Quy mô gốc: "1.000-4.999"        -> min/max: (1000, 4999)
  • Quy mô gốc: "1000+ nhân viên"   -> min/max: (1000, None)
  • URL riêng: "https://careers.fpt.com/about" -> domain: careers.fpt.com
  • URL sàn  : "https://www.topcv.vn/cong-ty/elva" -> domain: None


## 4. Xây Dựng Bảng `companies` & Khử Trùng Lặp Thực Thể (Entity Deduplication)

Quy tắc khử trùng lặp:
- Mỗi công ty được nhận diện duy nhất bằng tên chuẩn hóa: `company_name_normalized.lower()`.
- Cấp phát khóa chính tự tăng `company_id` (kiểu `BIGINT`).
- Giữ lại logo và website đại diện đầy đủ nhất nếu có nhiều tin cùng công ty.

In [10]:
company_map = {}
now_iso = datetime.now().isoformat()
company_id_counter = 1

for _, row in df_comp_merged.iterrows():
    raw_name = str(row.get('company_name_raw', '') or '').strip()
    norm_name = clean_company_name(raw_name)
    if not norm_name or norm_name == 'Unknown Company':
        continue
        
    comp_url = str(row.get('company_url', '') or '').strip()
    comp_logo = str(row.get('company_logo_url', '') or '').strip()
    comp_size_raw = str(row.get('company_size_raw', '') or '').strip()
    comp_loc_raw = str(row.get('company_location_raw', '') or '').strip()
    loc_raw = str(row.get('location_raw', '') or '').strip()
    ind_raw = str(row.get('industry_raw', '') or '').strip()
    
    domain = extract_website_domain(comp_url)
    comp_key = norm_name.lower()
    
    if comp_key not in company_map:
        size_min, size_max = parse_company_size(comp_size_raw)
        city = normalize_company_city(loc_raw, comp_loc_raw)
        
        company_map[comp_key] = {
            'company_id': company_id_counter,
            'company_name_raw': raw_name,
            'company_name_normalized': norm_name,
            'website_domain': domain,
            'company_url': comp_url,
            'company_logo_url': comp_logo,
            'company_size_raw': comp_size_raw,
            'company_size_min': size_min,
            'company_size_max': size_max,
            'company_location_raw': comp_loc_raw if comp_loc_raw else loc_raw,
            'company_city': city,
            'industry_raw': ind_raw,
            'industry_normalized': ind_raw if ind_raw else 'Công nghệ thông tin / IT',
            'created_at': now_iso,
            'updated_at': now_iso
        }
        company_id_counter += 1
    else:
        # Bổ sung các thông tin còn thiếu nếu tin sau có dữ liệu tốt hơn
        existing = company_map[comp_key]
        if not existing['company_logo_url'] and comp_logo:
            existing['company_logo_url'] = comp_logo
        if not existing['company_url'] and comp_url:
            existing['company_url'] = comp_url
            existing['website_domain'] = domain
        if existing['company_size_min'] is None and comp_size_raw:
            s_min, s_max = parse_company_size(comp_size_raw)
            existing['company_size_min'] = s_min
            existing['company_size_max'] = s_max
            existing['company_size_raw'] = comp_size_raw

df_companies = pd.DataFrame(list(company_map.values()))
print(f'✅ Khử trùng lặp thành công:')
print(f'  • Từ {len(df_comp_merged):,} tin tuyển dụng thô')
print(f'  • Gom về {len(df_companies):,} công ty độc nhất (Giảm {len(df_comp_merged) - len(df_companies):,} bản ghi trùng lặp)!')

✅ Khử trùng lặp thành công:
  • Từ 2,327 tin tuyển dụng thô
  • Gom về 1,204 công ty độc nhất (Giảm 1,123 bản ghi trùng lặp)!


## 5. Đánh Giá Chất Lượng Bảng `companies` Sau Làm Sạch & Preview Mẫu

Kiểm tra chất lượng và xem trước bảng `companies` hoàn chỉnh sẵn sàng nạp vào PostgreSQL.

In [12]:
# 1. Thông tin cấu trúc bảng companies
print('--- 1. CẤU TRÚC BẢNG companies (INFO) ---')
df_companies.info()

# 2. Thống kê thành phố đặt trụ sở
print('\n--- 2. PHÂN BỐ CÔNG TY THEO TỈNH/THÀNH PHỐ (company_city) ---')
display(df_companies['company_city'].value_counts())

# 3. Thống kê quy mô công ty
print('\n--- 3. PHÂN BỐ QUY MÔ NHÂN VIÊN TỐI THIỂU (company_size_min) ---')
size_bins = pd.cut(df_companies['company_size_min'].dropna(), bins=[0, 20, 50, 100, 500, 1000, 10000], labels=['< 20', '20-50', '50-100', '100-500', '500-1000', '> 1000'])
display(size_bins.value_counts())

# 4. Preview mẫu 10 dòng của bảng companies hoàn chỉnh
print('\n--- 4. PREVIEW 10 DÒNG BẢNG companies HOÀN CHỈNH (SẴN SÀNG ĐẨY POSTGRESQL) ---')
preview_cols = ['company_id', 'company_name_raw', 'company_name_normalized', 'company_size_min', 'company_size_max', 'company_city', 'industry_normalized']
display(df_companies[preview_cols].head(10))


--- 1. CẤU TRÚC BẢNG companies (INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 1204 entries, 0 to 1203
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   company_id               1204 non-null   int64  
 1   company_name_raw         1204 non-null   str    
 2   company_name_normalized  1204 non-null   str    
 3   website_domain           0 non-null      object 
 4   company_url              1204 non-null   str    
 5   company_logo_url         1204 non-null   str    
 6   company_size_raw         1204 non-null   str    
 7   company_size_min         979 non-null    float64
 8   company_size_max         896 non-null    float64
 9   company_location_raw     1204 non-null   str    
 10  company_city             1204 non-null   str    
 11  industry_raw             1204 non-null   str    
 12  industry_normalized      1204 non-null   str    
 13  created_at               1204 non-null   str   